# 07 --- End-to-End Pipeline

This notebook demonstrates the **full PINGU pipeline** from synthetic
wideband IQ generation to a geolocated position estimate.  It ties
together every stage of the processing chain:

1. **Channelization** -- splitting wideband IQ into narrowband channels.
2. **Detection** -- identifying signals of interest.
3. **TDoA estimation** -- cross-correlating receiver pairs.
4. **Bayesian integration** -- Kalman-filtering TDoA measurements.
5. **Convergence monitoring** -- detecting filter stabilisation.
6. **Position solving** -- weighted NLLS with Levenberg-Marquardt.

We exercise the pipeline on synthetic scenarios with varying SNR and
modulation types to characterise end-to-end performance.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pingu.config import load_config
from pingu.pipeline.runner import PinguPipeline
from pingu.synthetic.scenarios import TDoAScenario
from pingu.types import ReceiverConfig, ModulationType
from pingu.visualization.map_plot import plot_position_map
from pingu.visualization.convergence import plot_convergence
from pingu.locator.posterior import position_uncertainty

# Reproducible random generator
rng = np.random.default_rng(seed=42)

## Setup

Configure a 5-receiver pentagon array (100 km radius) and a transmitter
at a known position.  Load the default PINGU configuration and
instantiate the pipeline.

In [ ]:
# Build 5 receivers in a regular pentagon (100 km radius)
RADIUS = 100_000.0  # 100 km
N_RX = 5

receivers = []
for i in range(N_RX):
    angle = np.pi / 2 + 2 * np.pi * i / N_RX
    receivers.append(
        ReceiverConfig(
            id=f"RX{i}",
            latitude=40.0 + 0.3 * np.sin(angle),
            longitude=-75.0 + 0.4 * np.cos(angle),
            x=RADIUS * np.cos(angle),
            y=RADIUS * np.sin(angle),
            sample_rate=48_000.0,
        )
    )

# True transmitter position
TRUE_TX = (30_000.0, -20_000.0)  # (x, y) in metres

# Load default config and create the pipeline
config = load_config()
pipeline = PinguPipeline(config=config, receivers=receivers)

print("Receiver IDs:", [r.id for r in receivers])
print(f"Number of pairs: {pipeline.pair_manager.n_pairs}")

## Generate Synthetic Scenario

The `TDoAScenario` generates per-receiver IQ frames with realistic
propagation delays and independent AWGN noise.  We create 50 frames
(independent noise realisations) with BPSK modulation at 20 dB SNR.

In [ ]:
# Create TDoA scenario
scenario = TDoAScenario(
    receivers=receivers,
    tx_position=TRUE_TX,
    sample_rate=48_000.0,
    snr_db=20.0,
    duration=0.1,
    modulation=ModulationType.BPSK,
)

# Generate 50 independent frames
N_FRAMES = 50
scenario_frames = [scenario.generate(rng=rng) for _ in range(N_FRAMES)]

# Inspect the first frame
first_frame = scenario_frames[0]
sample_frame = first_frame[receivers[0].id]

print(f"Number of frames: {len(scenario_frames)}")
print(f"Samples per frame: {len(sample_frame.samples)}")
print(f"Sample rate: {sample_frame.sample_rate} Hz")
print(f"Duration per frame: {len(sample_frame.samples) / sample_frame.sample_rate:.3f} s")

## Run the Pipeline

Feed the scenario frames through the full pipeline.  The pipeline
processes each frame sequentially: channelise, detect, estimate TDoAs,
update the Kalman filter, check convergence, and solve for position.

In [ ]:
# Run the pipeline
result = pipeline.run(scenario_frames)

if result is not None:
    est_pos = np.array([result.x, result.y])
    true_pos = np.array(TRUE_TX)
    error_m = np.linalg.norm(est_pos - true_pos)
    n_updates = pipeline.kalman.n_updates

    print(f"Converged: Yes")
    print(f"Updates processed: {n_updates}")
    print(f"Estimated position: ({result.x:.1f}, {result.y:.1f}) m")
    print(f"True position:      ({TRUE_TX[0]:.1f}, {TRUE_TX[1]:.1f}) m")
    print(f"Position error:     {error_m:.1f} m ({error_m / 1e3:.3f} km)")
    print(f"95% confidence radius: {result.confidence_radius_95:.1f} m")
else:
    print("Pipeline did not converge.")

## Visualize Results

Use the built-in visualisation utilities to display the position map
with a confidence ellipse and the convergence of the Kalman filter
variances over time.

In [ ]:
# Position map with confidence ellipse
fig = plot_position_map(
    receivers=receivers,
    estimate=result,
    true_pos=TRUE_TX,
)
plt.show()

In [ ]:
# Convergence plot: variance history from the convergence monitor
variance_history = pipeline.convergence_monitor.get_variance_history()

fig = plot_convergence(
    variance_history=variance_history,
    title="TDoA Variance Convergence Over Pipeline Updates",
)
plt.show()

print(f"Variance reduction ratio: {pipeline.convergence_monitor.get_improvement_ratio():.1f}x")

## SNR Sensitivity

We sweep over a range of SNR values to characterise how pipeline
accuracy degrades with decreasing signal-to-noise ratio.

In [ ]:
# SNR sensitivity sweep
snr_values = [30, 20, 15, 10, 5]
snr_errors = []

print(f"{'SNR (dB)':>10s}  {'Error (m)':>12s}  {'Converged':>10s}")
print("-" * 38)

for snr in snr_values:
    # Create a fresh scenario and pipeline for each SNR
    scen = TDoAScenario(
        receivers=receivers,
        tx_position=TRUE_TX,
        sample_rate=48_000.0,
        snr_db=float(snr),
        duration=0.1,
        modulation=ModulationType.BPSK,
    )
    frames = [scen.generate(rng=rng) for _ in range(N_FRAMES)]

    pip = PinguPipeline(config=config, receivers=receivers)
    res = pip.run(frames)

    if res is not None:
        err = np.linalg.norm(np.array([res.x, res.y]) - np.array(TRUE_TX))
        snr_errors.append(err)
        print(f"{snr:10d}  {err:12.1f}  {'Yes':>10s}")
    else:
        snr_errors.append(np.nan)
        print(f"{snr:10d}  {'N/A':>12s}  {'No':>10s}")

# Plot error vs SNR
fig, ax = plt.subplots(figsize=(8, 5))
valid = ~np.isnan(snr_errors)
ax.semilogy(np.array(snr_values)[valid], np.array(snr_errors)[valid],
            "o-", linewidth=2, markersize=8)
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Position Error (m)")
ax.set_title("Position Error vs. SNR")
ax.grid(True, alpha=0.3, which="both")
ax.invert_xaxis()  # Higher SNR on the left
fig.tight_layout()
plt.show()

## Different Modulation Types

TDoA accuracy depends on the signal bandwidth -- broadband modulations
like BPSK and QPSK produce sharper cross-correlation peaks than
narrowband signals like CW.  We compare performance across five
modulation types at 20 dB SNR.

In [ ]:
# Modulation comparison
modulations = [
    ModulationType.CW,
    ModulationType.BPSK,
    ModulationType.QPSK,
    ModulationType.AM,
    ModulationType.SSB,
]
SNR_MOD = 20.0

print(f"{'Modulation':<12s}  {'Error (m)':>12s}  {'95% Radius (m)':>15s}  {'Converged':>10s}")
print("-" * 58)

for mod in modulations:
    scen = TDoAScenario(
        receivers=receivers,
        tx_position=TRUE_TX,
        sample_rate=48_000.0,
        snr_db=SNR_MOD,
        duration=0.1,
        modulation=mod,
    )
    frames = [scen.generate(rng=rng) for _ in range(N_FRAMES)]

    pip = PinguPipeline(config=config, receivers=receivers)
    res = pip.run(frames)

    if res is not None:
        err = np.linalg.norm(np.array([res.x, res.y]) - np.array(TRUE_TX))
        print(f"{mod.name:<12s}  {err:12.1f}  {res.confidence_radius_95:15.1f}  {'Yes':>10s}")
    else:
        print(f"{mod.name:<12s}  {'N/A':>12s}  {'N/A':>15s}  {'No':>10s}")

print()
print("Broadband modulations (BPSK, QPSK) generally provide better TDoA")
print("accuracy than narrowband signals (CW) due to sharper correlation peaks.")

## Summary

The PINGU pipeline processes wideband IQ data through all six stages --
channelization, detection, TDoA estimation, Bayesian integration,
convergence monitoring, and position solving -- to produce a geolocated
transmitter position with calibrated uncertainty.

Key observations:

- **SNR**: Position accuracy degrades gracefully as SNR decreases.
  The Kalman filter integrates multiple noisy observations to improve
  the final estimate.
- **Bandwidth**: Broadband modulations (BPSK, QPSK) yield sharper
  cross-correlation peaks and hence more precise TDoA estimates than
  narrowband signals (CW).
- **Geometry**: Receiver array geometry (baseline length and angular
  diversity) is the primary determinant of achievable position accuracy.